# Pipeline completo en Colab
Ingesta → perfiles → modelo de riesgo → cartografía → agente de intervención → app. Descarga los datos públicos del INE (~3.5 GB) y la cartografía. Ejecuta las celdas en orden.

In [ ]:
!git clone https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd ParticipacionCiudadanaINE
!pip -q install -r requirements.txt

Cloning into 'ParticipacionCiudadanaINE'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 43 (delta 10), reused 42 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 2.33 MiB | 23.15 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/ParticipacionCiudadanaINE


In [ ]:
%%bash
set -e
mkdir -p data/raw/zips && cd data/raw/zips
for p in 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2009.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2012.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2015.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2018.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2021.zip 2025/08/DatosAbiertos_DECEyEC_ConteosCensales2024.zip; do
  curl -sL -A 'Mozilla/5.0' -O "https://ine.mx/wp-content/uploads/$p"
done
for z in *.zip; do y=$(echo $z | grep -oE '[0-9]{4}'); mkdir -p ../$y && unzip -oq "$z" -d ../$y; done
rm -f *.zip
cd ../../.. && mkdir -p data/geo && cd data/geo
for f in SECCION MUNICIPIO ENTIDAD; do curl -sL -A 'Mozilla/5.0' -O "https://pautas.ine.mx/transparencia/mapas/cob_2025/$f.zip" && mkdir -p $f && unzip -oq $f.zip -d $f && rm $f.zip; done
ls ../raw ../geo

In [ ]:
!python -m src.ingest

In [ ]:
!python -W ignore -m src.profiles --objetivo 2024

In [ ]:
!python -W ignore -m src.risk_model

## Fase 4: cartografía ligera, agente y app

In [ ]:
!python -W ignore -m src.geo

In [ ]:
!python -m src.agent --anio 2027 --presupuesto-pct 10

### App (Streamlit)
Abre la URL `https://….trycloudflare.com` que se imprime al final. Para detenerla: *Entorno de ejecución → Interrumpir*.

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
sleep 12
grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1

In [ ]:
!cat data/processed/riesgo_metricas.json | head -c 6000

In [ ]:
!zip -qr processed.zip data/processed
from google.colab import files
files.download('processed.zip')